# Titanic 
## Score: .80382

In [24]:
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd()
DATA = ROOT / "titanic"
RARE = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}


In [25]:
def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()

    ticket_vc = pd.concat([tr["Ticket"], te["Ticket"]], ignore_index=True).value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]

    for pcl in (1, 2, 3):
        m = tr.loc[tr["Pclass"] == pcl, "Fare"].median()
        if pd.notna(m):
            tr.loc[tr["Pclass"] == pcl, "Fare"] = tr.loc[tr["Pclass"] == pcl, "Fare"].fillna(m)
            te.loc[te["Pclass"] == pcl, "Fare"] = te.loc[te["Pclass"] == pcl, "Fare"].fillna(m)
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        pcl = pd.to_numeric(out["Pclass"], errors="coerce").fillna(3).astype(int)
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["SexFemale"] = (out["Sex"] == "female").astype(int)
        out["IsChild"] = (out["Age"] < 16).astype(int)
        out["FemaleOrChild"] = ((out["Sex"] == "female") | (out["Age"] < 16)).astype(int)
        out["Deck"] = out["Cabin"].apply(lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U")
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgePclass"] = out["Age"] * pcl.astype(float)
        out["AgeBin"] = pd.cut(out["Age"], bins=[0, 12, 18, 35, 60, 200], labels=["c", "t", "y", "a", "s"]).astype(str)
        out["Surname"] = out["Name"].str.split(",").str[0].str.strip()
        out["TicketPrefix"] = out["Ticket"].astype(str).str.replace(r"[0-9./ ]", "", regex=True)
        out["TicketPrefix"] = out["TicketPrefix"].replace("", "NONE")
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin", "Surname", "TicketPrefix"]:
            out[c] = out[c].astype(str)
        return out

    X = feats(tr)
    X_test = feats(te)
    return X, y, X_test


def add_oof_target_encoding(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, cols, n_splits=5, alpha=15.0):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y_s = pd.Series(y)
    global_mean = y_s.mean()
    X_new = X.copy()
    X_test_new = X_test.copy()

    for col in cols:
        oof_col = np.zeros(len(X), dtype=float)
        for tr_i, va_i in cv.split(X, y):
            tr_c = X.iloc[tr_i][col]
            y_tr = y_s.iloc[tr_i]
            stat = y_tr.groupby(tr_c).agg(["mean", "count"])
            smooth = (stat["mean"] * stat["count"] + global_mean * alpha) / (stat["count"] + alpha)
            oof_col[va_i] = X.iloc[va_i][col].map(smooth).fillna(global_mean).to_numpy()
        full_stat = y_s.groupby(X[col]).agg(["mean", "count"])
        full_smooth = (full_stat["mean"] * full_stat["count"] + global_mean * alpha) / (full_stat["count"] + alpha)
        X_new[col + "_te"] = oof_col
        X_test_new[col + "_te"] = X_test[col].map(full_smooth).fillna(global_mean).to_numpy()
    return X_new, X_test_new


train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
pid = test_raw["PassengerId"].values
X, y, X_test = build_xy(train_raw, test_raw)
cat_cols = ["Sex", "Embarked", "Title", "Deck", "Pclass", "AgeBin", "Surname", "TicketPrefix"]
te_cols = ["Title", "Deck", "Embarked", "Pclass", "AgeBin", "Surname", "TicketPrefix"]
X_te, X_test_te = add_oof_target_encoding(X, y, X_test, te_cols, n_splits=5, alpha=15.0)

hgb_features = [
    "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "AgePclass",
    "TicketGroupSize", "FarePerPerson", "HasCabin",
    "SexFemale", "IsChild", "FemaleOrChild",
] + [c + "_te" for c in te_cols]

imp = SimpleImputer(strategy="median")
X_hgb = imp.fit_transform(X_te[hgb_features])
X_test_hgb = imp.transform(X_test_te[hgb_features])


In [26]:
def run_cv():
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_cb1 = np.zeros(len(y), dtype=float)
    oof_cb2 = np.zeros(len(y), dtype=float)
    oof_hgb = np.zeros(len(y), dtype=float)
    p_cb1_t = np.zeros(len(X_test), dtype=float)
    p_cb2_t = np.zeros(len(X_test), dtype=float)
    p_hgb_t = np.zeros(len(X_test), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        Xc_tr, Xc_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        Xh_tr, Xh_va = X_hgb[tr_idx], X_hgb[va_idx]

        cb1 = CatBoostClassifier(
            iterations=20000,
            learning_rate=0.016,
            depth=7,
            l2_leaf_reg=3.0,
            random_seed=SEED + fold,
            thread_count=1,
            verbose=False,
        )
        cb1.fit(Xc_tr, y_tr, cat_features=cat_cols, eval_set=(Xc_va, y_va), early_stopping_rounds=320, verbose=False)

        cb2 = CatBoostClassifier(
            iterations=26000,
            learning_rate=0.012,
            depth=9,
            l2_leaf_reg=4.5,
            random_seed=SEED + 100 + fold,
            thread_count=1,
            verbose=False,
        )
        cb2.fit(Xc_tr, y_tr, cat_features=cat_cols, eval_set=(Xc_va, y_va), early_stopping_rounds=380, verbose=False)

        hgb = HistGradientBoostingClassifier(
            learning_rate=0.02,
            max_iter=1400,
            max_depth=8,
            min_samples_leaf=6,
            l2_regularization=0.35,
            random_state=SEED + fold,
        )
        hgb.fit(Xh_tr, y_tr)

        oof_cb1[va_idx] = cb1.predict_proba(Xc_va)[:, 1]
        oof_cb2[va_idx] = cb2.predict_proba(Xc_va)[:, 1]
        oof_hgb[va_idx] = hgb.predict_proba(Xh_va)[:, 1]

        p_cb1_t += cb1.predict_proba(X_test)[:, 1] / cv.n_splits
        p_cb2_t += cb2.predict_proba(X_test)[:, 1] / cv.n_splits
        p_hgb_t += hgb.predict_proba(X_test_hgb)[:, 1] / cv.n_splits

    best_raw_acc, best_w1, best_w2, best_t_raw = -1.0, 0.33, 0.33, 0.5
    for w1 in np.linspace(0.22, 0.88, 67):
        for w2 in np.linspace(0.05, 0.58, 54):
            if w1 + w2 >= 0.98:
                continue
            w3 = 1.0 - w1 - w2
            p = w1 * oof_cb1 + w2 * oof_cb2 + w3 * oof_hgb
            for t in np.linspace(0.40, 0.62, 111):
                a = accuracy_score(y, (p >= t).astype(int))
                if a > best_raw_acc:
                    best_raw_acc, best_w1, best_w2, best_t_raw = a, float(w1), float(w2), float(t)
    best_w3 = 1.0 - best_w1 - best_w2
    oof_lin = best_w1 * oof_cb1 + best_w2 * oof_cb2 + best_w3 * oof_hgb
    test_lin = best_w1 * p_cb1_t + best_w2 * p_cb2_t + best_w3 * p_hgb_t

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof_lin, y)
    oof_cal = iso.predict(oof_lin)
    test_cal = iso.predict(test_lin)

    best_iso_acc, best_t_iso = -1.0, 0.5
    for t in np.linspace(0.05, 0.95, 181):
        a = accuracy_score(y, (oof_cal >= t).astype(int))
        if a > best_iso_acc:
            best_iso_acc, best_t_iso = a, float(t)

    use_iso = best_iso_acc >= best_raw_acc
    if use_iso:
        oof_final, test_final, t_final, acc_final = oof_cal, test_cal, best_t_iso, best_iso_acc
        branch = "isotonic"
    else:
        oof_final, test_final, t_final, acc_final = oof_lin, test_lin, best_t_raw, best_raw_acc
        branch = "raw_blend"

    print(
        "cv v8",
        branch,
        "acc",
        round(acc_final, 5),
        "t",
        round(t_final, 3),
        "w",
        [round(best_w1, 3), round(best_w2, 3), round(best_w3, 3)],
        "raw_acc",
        round(best_raw_acc, 5),
        "iso_acc",
        round(best_iso_acc, 5),
    )
    print(confusion_matrix(y, (oof_final >= t_final).astype(int)))

    return {
        "mode": f"v8_{branch}",
        "acc": acc_final,
        "t": t_final,
        "w": (best_w1, best_w2, best_w3),
        "oof_prob": oof_final,
        "test_prob": test_final,
        "iso": iso if use_iso else None,
    }


safe = run_cv()

for cutoff in (12, 14, 16):
    m = train_raw["Age"].fillna(999) < cutoff
    rate = float(train_raw.loc[m, "Survived"].mean()) if m.any() else float("nan")
    print(f"train survival age<{cutoff}: n={int(m.sum())}, rate={rate:.3f}")


cv v8 isotonic acc 0.85297 t 0.425 w [0.37, 0.51, 0.12] raw_acc 0.85297 iso_acc 0.85297
[[516  33]
 [ 98 244]]
train survival age<12: n=68, rate=0.574
train survival age<14: n=71, rate=0.592
train survival age<16: n=83, rate=0.590


In [27]:
prob = safe["test_prob"].copy()
pred = (prob >= safe["t"]).astype(int)

submission = pd.DataFrame({"PassengerId": pid, "Survived": pred})
submission.to_csv(ROOT / "submission.csv", index=False)
print("wrote", ROOT / "submission.csv", safe.get("mode", ""))
submission.head(10)


wrote c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv v8_isotonic


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
